# ML Assignment 4 – Regression & Evaluation Metrics
### Dataset: California Housing (sklearn)

**Objective:** Apply multiple regression algorithms to the California Housing dataset, evaluate their performance using standard metrics, perform cross-validation, tune hyperparameters, and identify the best model.

---
## Table of Contents
1. [Data Loading and Preprocessing](#section1)
2. [Regression Algorithm Implementation](#section2)
3. [Model Evaluation and Comparison](#section3)
4. [Cross-Validation and Hyperparameter Tuning](#section4)
5. [Selecting the Best Regression Model](#section5)

In [ ]:
# ── Standard Library & Utilities ──────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── sklearn: Data & Preprocessing ─────────────────────────────────────────────
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ── sklearn: Regression Models ─────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# ── sklearn: Metrics ───────────────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print('All libraries imported successfully.')

---
<a id='section1'></a>
## 1. Data Loading and Preprocessing

### 1.1 Loading the Dataset

In [ ]:
# Load California Housing dataset
housing = fetch_california_housing(as_frame=True)

# Build DataFrame and add target column
df = housing.frame

print('Shape:', df.shape)
print('\nFeature Descriptions:')
for name, desc in zip(housing.feature_names, housing.feature_names):
    print(f'  {name}')
print(f'  Target: MedHouseVal (Median House Value in $100,000s)')
df.head()

### 1.2 Preprocessing

#### Missing Value Check

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)
print(f'\nTotal missing values: {missing.sum()}')
print('\n✅ No missing values — no imputation required for the California Housing dataset.')

#### 1.3 Exploratory Data Analysis (EDA)

In [ ]:
# Descriptive statistics
df.describe().round(2)

In [ ]:
# Distribution of all features
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=40, edgecolor='white', color='steelblue')
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
plt.suptitle('Feature Distributions – California Housing', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 7))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix – California Housing', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop correlations with MedHouseVal (target):')
print(corr['MedHouseVal'].drop('MedHouseVal').sort_values(ascending=False))

In [ ]:
# Scatter plots for top correlated features vs. target
top_features = corr['MedHouseVal'].drop('MedHouseVal').abs().nlargest(4).index.tolist()
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, feat in zip(axes, top_features):
    ax.scatter(df[feat], df['MedHouseVal'], alpha=0.2, s=5, color='steelblue')
    ax.set_xlabel(feat)
    ax.set_ylabel('MedHouseVal')
    ax.set_title(f'{feat} vs Target')
plt.suptitle('Top Features vs. Median House Value', fontweight='bold')
plt.tight_layout()
plt.show()

#### 1.4 Feature Scaling

> **Choice: StandardScaler (Z-score Standardization)**  
> Standardization (μ=0, σ=1) is chosen over Min-Max normalization because:  
> - SVR and Linear Regression are sensitive to feature magnitude; standardization prevents any single feature from dominating.  
> - The features have different scales (e.g., `MedInc` ~0–15 vs `Population` ~0–35,000).  
> - Standardization is robust to outliers compared to Min-Max scaling.  
> - Tree-based models (DecisionTree, RandomForest, GBM) are scale-invariant, but scaling doesn't hurt them.

In [ ]:
# Features (X) and Target (y)
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

# Train / Test Split (80 / 20, random_state=42 for reproducibility)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature Scaling – fit ONLY on training data to prevent data leakage
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')
print(f'Features         : {X_train.shape[1]}')
print('\n✅ Scaler fitted on training set only — no data leakage.')

---
<a id='section2'></a>
## 2. Regression Algorithm Implementation

### 2.1 Linear Regression

**How it works:** Linear Regression models the relationship between features and the target as a linear combination: ŷ = β₀ + β₁x₁ + … + βₙxₙ. It minimizes the Residual Sum of Squares (RSS) using the Ordinary Least Squares (OLS) method.

**Suitability:** Acts as a strong baseline. If the relationship between median income (the most correlated feature) and house value is approximately linear, this model may perform reasonably well.

In [ ]:
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
print('Linear Regression — Training complete.')
print(f'  Coefficients: {dict(zip(X.columns, lr.coef_.round(3)))}')

### 2.2 Decision Tree Regressor

**How it works:** Recursively partitions the feature space into regions using binary splits, minimizing MSE at each split. Predictions are the mean target value within each leaf node.

**Suitability:** Can capture non-linear relationships and feature interactions naturally. However, it is prone to overfitting without constraints.

In [ ]:
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train_sc, y_train)
y_pred_dt = dt.predict(X_test_sc)
print('Decision Tree — Training complete.')
print(f'  Tree depth: {dt.get_depth()}')
print(f'  Leaf nodes: {dt.get_n_leaves()}')

### 2.3 Random Forest Regressor

**How it works:** An ensemble of Decision Trees trained on bootstrap samples (bagging) with random feature subsets at each split. Predictions are averaged across all trees, reducing variance and overfitting.

**Suitability:** Robust to outliers, handles non-linear interactions well, and generally generalizes better than a single Decision Tree.

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)
y_pred_rf = rf.predict(X_test_sc)
print('Random Forest — Training complete.')
print(f'  Number of estimators: {rf.n_estimators}')

### 2.4 Gradient Boosting Regressor

**How it works:** Builds trees sequentially, where each new tree corrects the residual errors of the previous ensemble using gradient descent in function space. Combines weak learners into a strong predictor.

**Suitability:** Often achieves the best accuracy on tabular datasets. Handles mixed feature types and non-linearities effectively.

In [ ]:
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train_sc, y_train)
y_pred_gb = gb.predict(X_test_sc)
print('Gradient Boosting — Training complete.')
print(f'  Number of estimators: {gb.n_estimators}')

### 2.5 Support Vector Regressor (SVR)

**How it works:** Finds a hyperplane that fits within a margin (ε-tube) around the data, minimizing error while maximizing the margin. Uses kernel functions (RBF by default) to handle non-linear relationships.

**Suitability:** Works well in high-dimensional spaces and is effective when feature relationships are complex. Requires feature scaling (already done).

In [ ]:
svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svr.fit(X_train_sc, y_train)
y_pred_svr = svr.predict(X_test_sc)
print('SVR — Training complete.')
print(f'  Kernel: {svr.kernel}, C: {svr.C}, epsilon: {svr.epsilon}')

---
<a id='section3'></a>
## 3. Model Evaluation and Comparison

Metrics used:
- **MSE** (Mean Squared Error) — penalizes large errors more heavily
- **MAE** (Mean Absolute Error) — average absolute prediction error (same unit as target)
- **R²** (Coefficient of Determination) — proportion of variance explained (1.0 = perfect)

In [ ]:
models = {
    'Linear Regression'       : y_pred_lr,
    'Decision Tree'           : y_pred_dt,
    'Random Forest'           : y_pred_rf,
    'Gradient Boosting'       : y_pred_gb,
    'SVR'                     : y_pred_svr,
}

results = []
for name, y_pred in models.items():
    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    results.append({'Model': name, 'MSE': round(mse,4), 'RMSE': round(rmse,4),
                    'MAE': round(mae,4), 'R²': round(r2,4)})

results_df = pd.DataFrame(results).set_index('Model')
results_df = results_df.sort_values('R²', ascending=False)
print('═'*65)
print('MODEL EVALUATION RESULTS')
print('═'*65)
print(results_df.to_string())
print('═'*65)
print(f"\n🏆 Best Model   : {results_df['R²'].idxmax()} (R² = {results_df['R²'].max()})")
print(f"⚠️  Worst Model  : {results_df['R²'].idxmin()} (R² = {results_df['R²'].min()})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
colors = sns.color_palette('muted', len(results_df))

for ax, metric, better in zip(axes, ['MSE', 'MAE', 'R²'], ['lower', 'lower', 'higher']):
    vals = results_df[metric].sort_values(ascending=(better=='lower'))
    bars = ax.barh(vals.index, vals.values, color=colors)
    ax.set_title(f'{metric} ({better} is better)', fontweight='bold')
    ax.set_xlabel(metric)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)

plt.suptitle('Model Comparison – Evaluation Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted plots for each model
fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=True)
for ax, (name, y_pred) in zip(axes, models.items()):
    ax.scatter(y_test, y_pred, alpha=0.2, s=5, color='steelblue')
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect')
    ax.set_title(name, fontweight='bold', fontsize=9)
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    r2 = r2_score(y_test, y_pred)
    ax.text(0.05, 0.92, f'R²={r2:.3f}', transform=ax.transAxes, fontsize=9,
            color='darkred', fontweight='bold')
plt.suptitle('Actual vs. Predicted – All Models', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.1 Analysis

| Model | Observation |
|---|---|
| **Gradient Boosting** | Typically best R² — captures complex non-linear patterns through sequential boosting |
| **Random Forest** | Very close to GB — ensemble averaging reduces variance effectively |
| **SVR** | Moderate performance; sensitive to C and epsilon hyperparameters |
| **Linear Regression** | Decent baseline; limited by linearity assumption |
| **Decision Tree** | Often worst due to overfitting — high variance without pruning |

---
<a id='section4'></a>
## 4. Cross-Validation and Hyperparameter Tuning

### 4.1 K-Fold Cross-Validation (k=5)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

base_models = {
    'Linear Regression'  : LinearRegression(),
    'Decision Tree'      : DecisionTreeRegressor(random_state=42),
    'Random Forest'      : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting'  : GradientBoostingRegressor(n_estimators=100, random_state=42),
    'SVR'                : SVR(kernel='rbf'),
}

cv_results = []
print('Running 5-Fold Cross-Validation...\n')
for name, model in base_models.items():
    # Use scaled data for all models
    scores = cross_val_score(model, X_train_sc, y_train,
                             cv=kf, scoring='r2', n_jobs=-1)
    cv_results.append({
        'Model'   : name,
        'CV Mean R²' : round(scores.mean(), 4),
        'CV Std R²'  : round(scores.std(),  4),
        'Min R²'     : round(scores.min(),  4),
        'Max R²'     : round(scores.max(),  4),
    })
    print(f'{name:<25} | Mean R²: {scores.mean():.4f} ± {scores.std():.4f}')

cv_df = pd.DataFrame(cv_results).set_index('Model')
cv_df

In [ ]:
# Cross-validation comparison plot
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(cv_df.index, cv_df['CV Mean R²'],
        xerr=cv_df['CV Std R²'],
        color=sns.color_palette('muted', len(cv_df)),
        capsize=5, edgecolor='white')
ax.set_xlabel('Mean R² (5-Fold CV)')
ax.set_title('Cross-Validation R² Scores with Std Dev', fontweight='bold')
for i, (v, s) in enumerate(zip(cv_df['CV Mean R²'], cv_df['CV Std R²'])):
    ax.text(v + 0.005, i, f'{v:.4f} ±{s:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

### 4.2 Hyperparameter Tuning

We use **GridSearchCV** for smaller search spaces and **RandomizedSearchCV** for larger ones.

#### Linear Regression (Ridge regularization)

In [ ]:
from sklearn.linear_model import Ridge

# Ridge adds L2 regularization — alpha controls regularization strength
ridge_params = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}
ridge_gs = GridSearchCV(Ridge(), ridge_params, cv=5, scoring='r2', n_jobs=-1)
ridge_gs.fit(X_train_sc, y_train)

print(f'Best alpha       : {ridge_gs.best_params_["alpha"]}')
print(f'Best CV R²       : {ridge_gs.best_score_:.4f}')

#### Decision Tree – GridSearchCV

Key hyperparameters:
- `max_depth`: Controls tree depth (prevents overfitting)
- `min_samples_split`: Minimum samples required to split a node
- `min_samples_leaf`: Minimum samples in a leaf node

In [ ]:
dt_params = {
    'max_depth'        : [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
}
dt_gs = GridSearchCV(DecisionTreeRegressor(random_state=42),
                     dt_params, cv=5, scoring='r2', n_jobs=-1)
dt_gs.fit(X_train_sc, y_train)

print(f'Best params      : {dt_gs.best_params_}')
print(f'Best CV R²       : {dt_gs.best_score_:.4f}')

#### Random Forest – RandomizedSearchCV

Key hyperparameters:
- `n_estimators`: Number of trees (more = better, but slower)
- `max_depth`: Maximum depth per tree
- `max_features`: Number of features at each split
- `min_samples_split` / `min_samples_leaf`: Control overfitting

In [ ]:
from scipy.stats import randint

rf_params = {
    'n_estimators'     : randint(50, 300),
    'max_depth'        : [5, 10, 20, None],
    'max_features'     : ['sqrt', 'log2', 0.5],
    'min_samples_split': randint(2, 10),
    'min_samples_leaf' : randint(1, 5),
}
rf_rs = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_params, n_iter=20, cv=5, scoring='r2',
    random_state=42, n_jobs=-1
)
rf_rs.fit(X_train_sc, y_train)

print(f'Best params      : {rf_rs.best_params_}')
print(f'Best CV R²       : {rf_rs.best_score_:.4f}')

#### Gradient Boosting – RandomizedSearchCV

Key hyperparameters:
- `n_estimators`: Number of boosting stages
- `learning_rate`: Shrinks contribution of each tree (trade-off with n_estimators)
- `max_depth`: Depth of individual trees
- `subsample`: Fraction of samples used per tree (stochastic boosting)

In [ ]:
gb_params = {
    'n_estimators' : [100, 200, 300],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth'    : [3, 5, 7],
    'subsample'    : [0.7, 0.8, 1.0],
    'min_samples_split': [2, 5],
}
gb_rs = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=42),
    gb_params, n_iter=20, cv=5, scoring='r2',
    random_state=42, n_jobs=-1
)
gb_rs.fit(X_train_sc, y_train)

print(f'Best params      : {gb_rs.best_params_}')
print(f'Best CV R²       : {gb_rs.best_score_:.4f}')

#### SVR – GridSearchCV

Key hyperparameters:
- `C`: Regularization — higher C = less regularization, fits training data more closely
- `epsilon`: Width of the insensitive tube — smaller = more sensitive to errors
- `gamma`: RBF kernel coefficient — controls influence radius

In [ ]:
svr_params = {
    'C'      : [0.1, 1.0, 10.0, 100.0],
    'epsilon': [0.01, 0.1, 0.5],
    'gamma'  : ['scale', 'auto'],
}
svr_gs = GridSearchCV(SVR(kernel='rbf'), svr_params,
                      cv=5, scoring='r2', n_jobs=-1)
svr_gs.fit(X_train_sc, y_train)

print(f'Best params      : {svr_gs.best_params_}')
print(f'Best CV R²       : {svr_gs.best_score_:.4f}')

### 4.3 Tuned Models — Test Set Evaluation

In [ ]:
tuned_models = {
    'Ridge (Tuned)'            : ridge_gs.best_estimator_,
    'Decision Tree (Tuned)'    : dt_gs.best_estimator_,
    'Random Forest (Tuned)'    : rf_rs.best_estimator_,
    'Gradient Boosting (Tuned)': gb_rs.best_estimator_,
    'SVR (Tuned)'              : svr_gs.best_estimator_,
}

tuned_results = []
print(f'{'Model':<30} | {'MSE':>8} | {'MAE':>8} | {'R²':>8}')
print('─'*58)
for name, model in tuned_models.items():
    y_pred = model.predict(X_test_sc)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    tuned_results.append({'Model': name, 'MSE': round(mse,4),
                          'MAE': round(mae,4), 'R²': round(r2,4)})
    print(f'{name:<30} | {mse:>8.4f} | {mae:>8.4f} | {r2:>8.4f}')

tuned_df = pd.DataFrame(tuned_results).set_index('Model')
best_tuned = tuned_df['R²'].idxmax()
print(f'\n🏆 Best Tuned Model: {best_tuned} (R² = {tuned_df["R²"].max()})')

In [ ]:
# Before vs After Tuning R² comparison
before = results_df['R²'].values
after  = tuned_df['R²'].values
model_names = ['Linear\nRegression', 'Decision\nTree',
               'Random\nForest', 'Gradient\nBoosting', 'SVR']

x = np.arange(len(model_names))
w = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w/2, before, w, label='Before Tuning', color='steelblue', alpha=0.8)
b2 = ax.bar(x + w/2, after,  w, label='After Tuning',  color='darkorange', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylabel('R² Score')
ax.set_title('R² Before vs. After Hyperparameter Tuning', fontweight='bold')
ax.legend()
for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

---
<a id='section5'></a>
## 5. Selecting the Best Regression Model

In [ ]:
# Identify best tuned model and retrain if needed
best_model_name = tuned_df['R²'].idxmax()
best_model      = tuned_models[best_model_name]
y_pred_best     = best_model.predict(X_test_sc)

final_mse  = mean_squared_error(y_test, y_pred_best)
final_rmse = np.sqrt(final_mse)
final_mae  = mean_absolute_error(y_test, y_pred_best)
final_r2   = r2_score(y_test, y_pred_best)

print('═'*55)
print(f'  🏆  BEST MODEL: {best_model_name}')
print('═'*55)
print(f'  MSE  : {final_mse:.4f}')
print(f'  RMSE : {final_rmse:.4f}  (in $100,000 units)')
print(f'  MAE  : {final_mae:.4f}  (in $100,000 units)')
print(f'  R²   : {final_r2:.4f}')
print('═'*55)

In [ ]:
# Feature Importance (for tree-based best model)
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_,
                            index=X.columns).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(8, 4))
    importances.plot.bar(ax=ax, color=sns.color_palette('muted', len(importances)))
    ax.set_title(f'Feature Importances – {best_model_name}', fontweight='bold')
    ax.set_ylabel('Importance')
    ax.set_xlabel('Feature')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
    print('Top features:', importances.head(3).index.tolist())
else:
    print(f'{best_model_name} does not expose feature importances.')

In [ ]:
# Residual Analysis of Best Model
residuals = y_test - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residuals vs Predicted
axes[0].scatter(y_pred_best, residuals, alpha=0.2, s=5, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Predicted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs. Predicted', fontweight='bold')

# Residual distribution
axes[1].hist(residuals, bins=50, edgecolor='white', color='steelblue')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution', fontweight='bold')

plt.suptitle(f'Residual Analysis – {best_model_name}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Residual Mean : {residuals.mean():.4f}  (should be ≈ 0)')
print(f'Residual Std  : {residuals.std():.4f}')

---
## 5.1 Final Conclusion & Justification

### Why Gradient Boosting / Random Forest Wins

| Criterion | Gradient Boosting | Random Forest | Linear Regression | Decision Tree | SVR |
|---|---|---|---|---|---|
| **Handles Non-linearity** | ✅ Excellent | ✅ Excellent | ❌ Limited | ✅ Good | ✅ Good |
| **Overfitting Risk** | Low (regularized) | Low (bagging) | Very Low | High | Low |
| **Feature Interactions** | ✅ Captures naturally | ✅ Captures naturally | ❌ No | ✅ Yes | Partial |
| **Scalability** | Moderate | Fast (parallel) | Fast | Fast | Slow on large data |
| **Interpretability** | Moderate | Moderate | High | High | Low |

### Key Insights

1. **MedInc (Median Income)** is the single most important feature, confirming that income is the strongest predictor of housing prices.
2. **Geographic features** (`Latitude`, `Longitude`) also carry significant importance — California coastal vs inland price differences are large.
3. **Gradient Boosting** outperforms because: (a) it corrects errors iteratively, (b) it captures complex non-linear feature interactions, and (c) hyperparameter tuning (especially `learning_rate` + `n_estimators`) fine-tunes the bias-variance balance optimally.
4. **Decision Tree without pruning** overfits severely — it memorizes training data but fails to generalize.
5. **Linear Regression** provides a useful baseline but cannot capture the non-linear relationship between income/geography and house prices.

### Recommendation
> **Gradient Boosting Regressor** (tuned) is the recommended model for the California Housing dataset. It achieves the highest R² and lowest MSE/MAE, with residuals approximately centered at zero, indicating well-calibrated predictions.